# Lab 01 — Tokenization

**Course:** Fundamentals of NLP
**Audience:** M1 GAMMA, L2 LMAD Data Science
**Author:** Aymen Ben Brik

## Objectives

By the end of this lab, you will be able to:
1. Tokenize text at three granularities (word, character, sub-word).
2. Compare the trade-offs between vocabulary size, sequence length, and OOV rate.
3. Implement the Byte-Pair Encoding (BPE) algorithm from scratch.
4. Use modern pre-trained tokenizers (BERT, GPT-2) and inspect their behaviour.
5. Discuss multilingual and dialectal tokenization challenges.

## Prerequisites

```bash
pip install nltk transformers regex
```

## 1. Setup

In [ ]:
import re
from collections import Counter, defaultdict
import nltk
for pkg in ('punkt', 'punkt_tab'):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

## 2. Whitespace tokenization

The simplest tokenizer just splits on whitespace. Try it on a few sentences and notice the limitations: punctuation glued to words, contractions kept together, multiple spaces preserved.

In [ ]:
text = "I've been to Sfax-University; it's amazing! See you tomorrow."
tokens = text.split()
print('Whitespace tokens:', tokens)
print('Number of tokens:', len(tokens))

## 3. Whitespace + punctuation (regex)

In [ ]:
tokens = re.findall(r"\w+|[^\w\s]", text)
print('Regex tokens:', tokens)
print('Number of tokens:', len(tokens))

**Question.** What happens to *I've*? What happens to *Sfax-University*? Comment.

## 4. NLTK word tokenizer

In [ ]:
tokens = nltk.word_tokenize(text)
print('NLTK tokens:', tokens)
print('Number of tokens:', len(tokens))

Notice that NLTK splits *I've* into *I* + *'ve*, and treats *Sfax-University* differently from a regex split.

## 5. Character-level tokenization

In [ ]:
char_tokens = list(text)
print('First 30 chars:', char_tokens[:30])
print('Vocabulary size (unique chars):', len(set(char_tokens)))
print('Sequence length:', len(char_tokens))

**Trade-off observed.** The vocabulary is tiny (a few dozen distinct characters), but sequence length is much longer than at the word level. Models pay a memory and compute cost for character-level inputs.

## 6. Byte-Pair Encoding from scratch

We implement BPE on a small corpus and watch the merges grow.

In [ ]:
def get_pair_counts(vocab: dict) -> Counter:
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair: tuple, vocab: dict) -> dict:
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = pattern.sub(''.join(pair), word)
        new_vocab[new_word] = freq
    return new_vocab

def train_bpe(corpus_words: dict, num_merges: int) -> tuple:
    """Each key is a word represented as space-separated chars + </w>."""
    vocab = {' '.join(list(w)) + ' </w>': f for w, f in corpus_words.items()}
    merges = []
    for step in range(num_merges):
        pairs = get_pair_counts(vocab)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        merges.append(best)
        vocab = merge_vocab(best, vocab)
        print(f'Step {step + 1}: merge {best} (freq {pairs[best]})')
    return vocab, merges

corpus = {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}
vocab, merges = train_bpe(corpus, num_merges=10)

print('\nFinal vocabulary entries:')
for w, f in vocab.items():
    print(f'  {w:<20} freq={f}')

**Observation.** Frequent words (`low`, `newest`) are merged into single tokens. Rare combinations remain decomposed. The merge order is the BPE *training output*, used at inference to tokenize new words.

## 7. Pre-trained tokenizers (BERT vs GPT-2)

In [ ]:
# pip install transformers if needed
try:
    from transformers import AutoTokenizer
    bert = AutoTokenizer.from_pretrained('bert-base-uncased')
    gpt2 = AutoTokenizer.from_pretrained('gpt2')
    sentence = "Tokenization underlies every NLP system."
    print('BERT tokens:', bert.tokenize(sentence))
    print('GPT-2 tokens:', gpt2.tokenize(sentence))
except ImportError:
    print('Install transformers: pip install transformers')

**Discussion.**
- BERT uses *WordPiece*, marking sub-word continuations with `##`.
- GPT-2 uses byte-level BPE, with leading-space markers (`Ġ`).
Both produce different sub-word splits despite the same algorithmic family.

## 8. Multilingual / dialectal challenge

In [ ]:
dialectal = "n7eb el sufflé bil chocolat n3awd l week-end jay"  # Tunisian arabizi + French
try:
    print('BERT (English):', bert.tokenize(dialectal))
    multi = AutoTokenizer.from_pretrained('xlm-roberta-base')
    print('XLM-R (multilingual):', multi.tokenize(dialectal))
except (NameError, OSError):
    print('Skipping (transformers / model not available)')

**Observation.** A general English tokenizer fragments dialectal arabizi badly (digits like `7` become rare sub-tokens). A multilingual tokenizer like XLM-R does noticeably better.

## Exercises

**Exercise 1.** Tokenize the sentence "*The CEOs' meeting was rescheduled.*" with three different tokenizers (regex, NLTK, BERT). Comment on the differences for the words *CEOs'* and *rescheduled*.

**Exercise 2.** Train BPE on the toy corpus `{'xay': 1, 'yax': 1, 'xy': 1}` for 4 merges. List the new vocabulary at each step.

**Exercise 3.** For the GPT-2 tokenizer, count how many tokens the following English-vs-French sentence pair produces. Comment on cross-lingual fairness:
- *"The weather is nice today."*
- *"Le temps est agréable aujourd'hui."*

**Exercise 4.** Pick a Tunisian dialectal sentence of your own. Tokenize it with three different tokenizers and identify the strengths and weaknesses of each.

**Exercise 5 (synthesis).** Implement the *encoding* phase of BPE: given a list of merges learnt from a corpus, write a function that tokenizes a new (unseen) word by greedily applying the merges in order.